# 🔧 Certificate Classification - Part 2: Feature Engineering

## Goal
Transform raw certificate data into ML-ready features.

### Feature Categories
1. **Numeric**: key size, SAN count, validity days
2. **Categorical**: algorithms, key types
3. **Text-derived**: domain entropy, length, TLD
4. **Temporal**: certificate age, year issued
5. **Security flags**: weak crypto indicators

---

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import math
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Load explored data
pkl_path = Path('./outputs/ml/df_explored.pkl')
csv_path = Path('./outputs/ml/labeled_sheet.csv')

if pkl_path.exists():
    df = pd.read_pickle(pkl_path)
    print('Loaded from pickle')
else:
    df = pd.read_csv(csv_path, low_memory=False)
    print('Loaded from CSV')

print(f'Records: {len(df):,}')

## 2. Create Target Variable

In [ ]:
# Binary target: 1 = malicious, 0 = unknown/benign
df['y'] = (df['label'] == 'malicious').astype(int)

print('Target distribution:')
print(df['y'].value_counts())
print(f'\nPositive rate: {df["y"].mean()*100:.4f}%')

## 3. Numeric Features

In [ ]:
# 3.1 Public key size (numeric)
df['feat_key_size'] = pd.to_numeric(df['public_key_size'], errors='coerce').fillna(0).astype(int)

# 3.2 SAN count
df['feat_san_count'] = df['san_count'].fillna(0).astype(int)

# 3.3 Validity period
df['not_before_dt'] = pd.to_datetime(df['not_before'], errors='coerce')
df['not_after_dt'] = pd.to_datetime(df['not_after'], errors='coerce')
df['feat_validity_days'] = (df['not_after_dt'] - df['not_before_dt']).dt.days.fillna(0).astype(int)

# 3.4 Certificate age (days since not_before)
now = pd.Timestamp.now()
df['feat_age_days'] = (now - df['not_before_dt']).dt.days.fillna(0).astype(int)

# 3.5 Year issued
df['feat_year_issued'] = df['not_before_dt'].dt.year.fillna(2000).astype(int)

print('Numeric features created:')
print(df[['feat_key_size', 'feat_san_count', 'feat_validity_days', 'feat_age_days', 'feat_year_issued']].describe())

## 4. Security Flag Features

In [ ]:
# 4.1 Weak key (RSA < 2048)
df['feat_weak_key'] = ((df['feat_key_size'] > 0) & (df['feat_key_size'] < 2048)).astype(int)

# 4.2 SHA-1 signature (deprecated)
df['feat_sha1_sig'] = df['signature_hash_algo'].str.upper().str.contains('SHA-1|SHA1', na=False).astype(int)

# 4.3 SHA-256 signature (modern)
df['feat_sha256_sig'] = df['signature_hash_algo'].str.upper().str.contains('SHA-256|SHA256', na=False).astype(int)

# 4.4 RSA key type
df['feat_rsa_key'] = df['public_key_algo'].str.upper().str.contains('RSA', na=False).astype(int)

# 4.5 ECC key type (modern)
df['feat_ecc_key'] = df['public_key_algo'].str.upper().str.contains('EC|ECDSA', na=False).astype(int)

# 4.6 Very short validity (< 90 days) - suspicious
df['feat_short_validity'] = (df['feat_validity_days'] < 90).astype(int)

# 4.7 Very long validity (> 825 days / ~27 months) - old practice
df['feat_long_validity'] = (df['feat_validity_days'] > 825).astype(int)

print('Security flags:')
security_cols = [c for c in df.columns if c.startswith('feat_') and ('weak' in c or 'sha' in c or 'key' in c or 'validity' in c)]
print(df[security_cols].sum())

## 5. Domain Text Features

In [ ]:
def calc_entropy(s):
    """Calculate Shannon entropy of a string."""
    if not s or len(s) == 0:
        return 0.0
    freq = Counter(s.lower())
    probs = [c / len(s) for c in freq.values()]
    return -sum(p * math.log2(p) for p in probs if p > 0)

def extract_tld(domain):
    """Extract TLD from domain."""
    if not domain or not isinstance(domain, str):
        return 'unknown'
    parts = domain.lower().strip().rstrip('.').split('.')
    return parts[-1] if parts else 'unknown'

def count_dots(s):
    """Count dots in domain."""
    return s.count('.') if isinstance(s, str) else 0

def count_digits(s):
    """Count digits in domain."""
    return sum(c.isdigit() for c in s) if isinstance(s, str) else 0

def count_hyphens(s):
    """Count hyphens in domain."""
    return s.count('-') if isinstance(s, str) else 0

In [ ]:
# Apply domain features to common_name
cn = df['common_name'].fillna('')

# 5.1 Domain length
df['feat_cn_length'] = cn.str.len()

# 5.2 Domain entropy
df['feat_cn_entropy'] = cn.apply(calc_entropy)

# 5.3 Dot count (subdomain depth)
df['feat_cn_dots'] = cn.apply(count_dots)

# 5.4 Digit count
df['feat_cn_digits'] = cn.apply(count_digits)

# 5.5 Hyphen count
df['feat_cn_hyphens'] = cn.apply(count_hyphens)

# 5.6 TLD
df['feat_tld'] = cn.apply(extract_tld)

# 5.7 Is wildcard
df['feat_is_wildcard'] = cn.str.startswith('*').astype(int)

# 5.8 Contains 'www'
df['feat_has_www'] = cn.str.lower().str.contains('www', na=False).astype(int)

print('Domain features sample:')
print(df[['common_name', 'feat_cn_length', 'feat_cn_entropy', 'feat_cn_dots', 'feat_tld']].head(10))

## 6. Categorical Encoding

In [ ]:
# Encode top TLDs, others as 'other'
top_tlds = df['feat_tld'].value_counts().head(20).index.tolist()
df['feat_tld_enc'] = df['feat_tld'].apply(lambda x: x if x in top_tlds else 'other')

# One-hot encode TLD
tld_dummies = pd.get_dummies(df['feat_tld_enc'], prefix='tld')
df = pd.concat([df, tld_dummies], axis=1)

print(f'Created {len(tld_dummies.columns)} TLD features')

In [ ]:
# Encode signature algorithm
sig_dummies = pd.get_dummies(df['signature_hash_algo'].fillna('unknown'), prefix='sig')
df = pd.concat([df, sig_dummies], axis=1)

print(f'Created {len(sig_dummies.columns)} signature algorithm features')

## 7. Issuer Features

In [ ]:
# Extract issuer organization
def extract_issuer_org(issuer_dn):
    if not issuer_dn or not isinstance(issuer_dn, str):
        return 'unknown'
    match = re.search(r'O=([^,]+)', issuer_dn)
    if match:
        return match.group(1).strip().lower()[:50]
    return 'unknown'

df['feat_issuer_org'] = df['issuer_dn'].apply(extract_issuer_org)

# Top issuers
top_issuers = df['feat_issuer_org'].value_counts().head(15).index.tolist()
df['feat_issuer_enc'] = df['feat_issuer_org'].apply(lambda x: x if x in top_issuers else 'other')

print('Top issuers:')
print(df['feat_issuer_org'].value_counts().head(10))

## 8. Final Feature Set

In [ ]:
# Identify all feature columns
feature_cols = [c for c in df.columns if c.startswith('feat_') or c.startswith('tld_') or c.startswith('sig_')]

# Remove non-numeric feature columns (we'll handle them separately)
numeric_features = []
for col in feature_cols:
    if df[col].dtype in ['int64', 'float64', 'int32', 'float32', 'bool', 'int', 'uint8']:
        numeric_features.append(col)

print(f'Total numeric features: {len(numeric_features)}')
print('\nFeature list:')
for i, f in enumerate(sorted(numeric_features), 1):
    print(f'  {i:2}. {f}')

In [ ]:
# Create final feature matrix
X = df[numeric_features].copy()
y = df['y'].copy()

# Fill any remaining NaN
X = X.fillna(0)

print(f'Feature matrix shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'\nPositive samples: {y.sum()} ({y.mean()*100:.4f}%)')

In [ ]:
# Check feature correlations with target
correlations = X.corrwith(y).sort_values(ascending=False)
print('Top 15 features correlated with malicious:')
print(correlations.head(15))
print('\nBottom 10:')
print(correlations.tail(10))

## 9. Save Processed Data

In [ ]:
# Save for model training
output_dir = Path('./outputs/ml')

# Save feature matrix and target
X.to_pickle(output_dir / 'X_features.pkl')
y.to_pickle(output_dir / 'y_target.pkl')

# Save feature names
with open(output_dir / 'feature_names.txt', 'w') as f:
    f.write('\n'.join(numeric_features))

print('✅ Saved:')
print(f'   - X_features.pkl ({X.shape})')
print(f'   - y_target.pkl ({y.shape})')
print(f'   - feature_names.txt ({len(numeric_features)} features)')